# View Segmentation Results
Loads saved omnipose results from disk and opens them in napari.
No segmentation run needed — just load and view.

**Kernel:** `conda activate omnipose` before launching Jupyter.

In [ ]:
from pathlib import Path

# ── USER SETTINGS ────────────────────────────────────────────────────────────
_base      = Path.home() / "Box/Zohar_Persky/projects/p2f-revisions/morph-omnipose"
OUTPUT_DIR = _base / "omnipose_seg"

NAPARI_FOV = 0   # 0-based index, or set to a name string e.g. "fov_1_hyb_1"
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import numpy as np
import tifffile

fov_dirs = sorted(p for p in OUTPUT_DIR.iterdir()
                  if p.is_dir() and (p / f"{p.name}.seg.npy").exists())

print(f"Found {len(fov_dirs)} saved FOV(s) in {OUTPUT_DIR}:")
for i, d in enumerate(fov_dirs):
    print(f"  [{i}] {d.name}")

In [ ]:
if isinstance(NAPARI_FOV, int):
    fov_dir = fov_dirs[NAPARI_FOV]
else:
    fov_dir = OUTPUT_DIR / NAPARI_FOV

fov_name = fov_dir.name
mask = np.load(fov_dir / f"{fov_name}.seg.npy")
img  = tifffile.imread(fov_dir / f"{fov_name}.seg_image.tif")

print(f"Loaded: {fov_name}")
print(f"  image  shape={img.shape}  dtype={img.dtype}")
print(f"  mask   shape={mask.shape}  cells={mask.max()}")

In [ ]:
# ── Quick matplotlib preview ──────────────────────────────────────────────────
import matplotlib.pyplot as plt
from skimage.color import label2rgb

overlay = label2rgb(mask, image=img.astype(float) / img.max(), bg_label=0, alpha=0.35)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(img,     cmap="gray");  axes[0].set_title("Phase")
axes[1].imshow(mask,    cmap="tab20"); axes[1].set_title(f"Masks  ({mask.max()} cells)")
axes[2].imshow(overlay);               axes[2].set_title("Overlay")
for ax in axes: ax.axis("off")
fig.suptitle(fov_name, fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Napari viewer (interactive) ───────────────────────────────────────────────
import napari

viewer = napari.Viewer(title=fov_name)
viewer.add_image(img,  name="phase",          colormap="gray",
                 contrast_limits=[int(img.min()), int(img.max())])
viewer.add_labels(mask, name="omnipose_masks")
napari.run()